# Gemma 4 reasoning: recover failed and unfinished emails

The original reasoning-enabled run saved 214 of 559 predictions at temperature 0. Row 112 repeatedly exhausted 4,096 and 8,192 output tokens. The recovery preserves every successful historical row and requests only the missing IDs.

Recovery keeps reasoning enabled and **temperature at 0 and seed at 2** for every new request. It preserves the original request's omission of top-p and top-k overrides. Each email starts at 4,096 output tokens, with at most two larger-cap retries (8,192 and 16,384) for truncated responses. If these caps are exhausted, two final attempts on Darkbloom add `repetition_penalty=1.1` at caps 4,096 and 8,192 while keeping temperature 0 and seed 2. This is a bounded recovery policy; it does not silently turn reasoning off.

Every successful and rejected response is recorded in `data/gemma4_reasoning_recovery_attempts.jsonl`, including request ID, reported cost, token counts, and effective settings. Raw reasoning text is not stored. Completed IDs are skipped on resume. The combined output records the origin, sampling settings, token cap, and configuration hash for each prediction.

**Interpretation:** all results use temperature 0; recovered rows use seed 2 while historical rows did not specify a seed. Output-token caps can differ by row. They show completion and performance of this recovery workflow, not a controlled comparison that isolates reasoning. Historical failed and interrupted requests were not fully accounted for, so the recovery costs are reported separately.

## Completed recovery — October 1, 2026

All **559/559** predictions are available: 214 historical successes and 345 recovered rows. All new requests used temperature 0 and seed 2. Row 539 completed at an 8,192-token cap. Row 112 exhausted the normal cap schedule but succeeded with repetition penalty 1.1, using 244 reasoning tokens. Every final response reported reasoning tokens.

Recovery made 349 recorded attempts, including 4 truncated responses, for **$0.062392**. No request was sent to Google Vertex; the proposed provider change was blocked and the final repair used the previously authorized Darkbloom provider. The combined predictions have precision **47.9%**, recall **93.2%**, and F1 **63.3%**. Historical rows did not specify a seed, and one recovered row uses a repetition penalty, so this is a documented recovery result rather than one uniform-configuration benchmark.


In [1]:
import json
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/gemma_reasoning_recovery.py').exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from scripts.gemma_reasoning_recovery import run, summarize, source_rows, SUMMARY, OUTPUT
RUN_RECOVERY = False  # True sends only missing emails; key comes from environment or local .env.
if RUN_RECOVERY:
    run(pilot=True)
    run()
summary = summarize(source_rows())
print('Status:', summary['status'])
print('Completed:', summary['rows'], '/', summary['expected_rows'])
print('Preserved:', summary['legacy_rows'], '| Recovered:', summary['recovered_rows'])
print('Missing IDs:', summary['missing_source_rows'])
print('Known recovery cost: $', round(summary['recovery_known_cost_usd'], 6))
print('Results:', OUTPUT)


Status: complete
Completed: 559 / 559
Preserved: 214 | Recovered: 345
Missing IDs: []
Known recovery cost: $ 0.062392
Results: /Volumes/NVME/Source/GTPracticum/data/gemma4_include_reasoning_recovered_test_predictions.csv


## Full-set comparison

The table appears only after every test email has a saved prediction. The recovery total includes recorded failed responses; the metrics' prediction cost covers successful rows only. The old `gemma4_include_reasoning_enabled*` files remain unchanged for audit.


In [2]:
from scripts.confidence_comparison import read_csv
from scripts.gemma4_openrouter import evaluate
if summary['status'] == 'complete':
    runs = [('Gemma recovered reasoning', OUTPUT),
            ('Gemma reasoning disabled', REPO/'data/gemma4_include_no_reasoning_test_predictions.csv'),
            ('JEV', REPO/'data/jev_test_predictions.csv')]
    print(f"{'Run':<28} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FP':>5} {'FN':>5}")
    for name, path in runs:
        m = evaluate(read_csv(path))
        print(f"{name:<28} {m['precision']:>10.3f} {m['recall']:>8.3f} {m['f1']:>8.3f} {m['fp']:>5} {m['fn']:>5}")
else:
    print('Incomplete: full-set metrics are withheld.')


Run                           Precision   Recall       F1    FP    FN
Gemma recovered reasoning         0.479    0.932    0.633    74     5
Gemma reasoning disabled          0.462    0.904    0.611    77     7
JEV                               0.680    0.699    0.689    24    22
